# Aplicações

In [ ]:
import numpy as np
import scipy.integrate as spi
import matplotlib.pyplot as plt

### Sistema Massa-Mola-Amortecedor

Um sistema massa-mola-amortecedor é uma equação de segunda ordem com coeficientes constantes:

$$
my'' + cy' + ky = F(t)
$$

onde $m > 0, c \geq 0, k \geq 0$, e $F(t)$ é a função de força externa. Escreva a equação como um sistema de primeira ordem 2-dimensional introduzindo novas variáveis $u_0 = y$ e $u_1 = y'$:

\begin{align}
u'_0 &= u_1 \\\
u'_1 &= (F(t)-ku_0-cu_1)/m
\end{align}

Escreva uma função chamada `mass_spring_damper` que receba os parâmetros de entrada `m`, `c`, `k`, `F`, `u0` e `t` onde

* `m`, `c`, `k` são coeficientes do sistema
* `F` é a função de força externa $F(t)$
* `u0` é o vetor de condições iniciais `u0 = [y(0), y'(0)]`
* `t` é um array NumPy 1D

A função retorna um array `y` de valores que aproximam a solução $y(t)$ nos valores de $t$ dados por `t`.

In [ ]:
def mass_spring_damper(m,c,k,F,u0,t):

    def f(u,t):
        dudt = np.zeros(2)
        dudt[0] = u[1]
        dudt[1] = (F(t) - k*u[0] - c*u[1])/m
        return dudt

    U = spi.odeint(f,u0,t)
    y = U[:,0]
    return y

Se $m=1$, $c=0$, $k=1$, $F(t)=0$, $y(0)=0$, $y'(0)=1$, então a equação é

$$
y'' + y = 0
$$

para $y(0)=0$, $y'(0)=1$ e a solução é

$$
y=\sin(t)
$$

In [ ]:
m = 1; c = 0; k = 1;
F = lambda t: 0
t = np.linspace(0,4*np.pi,100)
u0 = [0,1]
y = mass_spring_damper(m,c,k,F,u0,t)
plt.plot(t,y,t,np.sin(t),'r.'), plt.grid(True)
plt.xlabel('$t$'), plt.ylabel('$y$'), plt.legend(['odeint',r'Exata: $\sin(t)$'])
plt.show()

A [ressonância](https://www.jirka.org/diffyqs/html/forcedo_section.html) ocorre quando não há amortecimento $c=0$ e a frequência da força externa é igual à frequência natural $\omega_n = \sqrt{k/m}$, ou seja $F(t)=\sin \left(\sqrt{\frac{k}{m}} t\right) $.

In [ ]:
m = 1; c = 0; k = 10;
F = lambda t: np.sin((k/m)**0.5*t)    # frequência igual à natural
t = np.linspace(0,50,1000)
u0 = [0,0]
y = mass_spring_damper(m,c,k,F,u0,t)
plt.plot(t,y), plt.grid(True)
plt.show()

O [batimento](https://www.jirka.org/diffyqs/html/forcedo_section.html) descreve o comportamento de um sistema não amortecido $c=0$ quando a frequência da força externa está próxima, mas não é igual à frequência natural $\omega_n = \sqrt{k/m}$.

In [ ]:
m = 1; c = 0; k = 10;
F = lambda t: np.sin(3*t)
t = np.linspace(0,100,1000)
u0 = [0,0]
y = mass_spring_damper(m,c,k,F,u0,t)
plt.plot(t,y), plt.grid(True)
plt.show()

Em um sistema amortecido ($c \neq 0$) não há ressonância no sentido de amplitude ilimitada. Para uma força $F(t) = \sin(\omega t)$, depois que o transiente desaparece, a solução oscila com amplitude

$$
A(\omega) = \frac{1}{\sqrt{(k - m\omega^2)^2 + (c\omega)^2}},
$$

que é máxima na frequência de **ressonância prática**

$$
\omega_r = \sqrt{\frac{k}{m} - \frac{c^2}{2m^2}}
$$

(quando $c^2 < 2mk$). Cuidado para não confundir com a **frequência natural amortecida** $\omega_d = \sqrt{\frac{k}{m} - \frac{c^2}{4m^2}}$, que é a frequência das oscilações livres ($F=0$). Para $m=1$, $c=2$, $k=10$, temos $\omega_r = \sqrt{8} \approx 2.83$ e $\omega_d = 3$. Veja a [ressonância prática](https://www.jirka.org/diffyqs/html/forcedo_section.html) nas notas de Lebl.

In [ ]:
m = 1; c = 2; k = 10;
wr = np.sqrt(k/m - c**2/(2*m**2))    # ressonância prática
F = lambda t: np.sin(wr*t)
t = np.linspace(0,15,200)
u0 = [0,0]
y = mass_spring_damper(m,c,k,F,u0,t)
plt.plot(t,y), plt.grid(True)
plt.show()

Podemos confirmar numericamente a curva $A(\omega)$: para cada frequência $\omega$, resolvemos a equação, descartamos o transiente e medimos a amplitude máxima da solução no final do intervalo.

In [ ]:
m = 1; c = 2; k = 10;
omegas = np.linspace(0.5,6,40)
A_num = np.zeros(len(omegas))
t = np.linspace(0,40,4000)
for i, w in enumerate(omegas):
    y = mass_spring_damper(m,c,k,lambda t: np.sin(w*t),[0,0],t)
    A_num[i] = np.max(np.abs(y[t > 30]))       # amplitude após o transiente

w_fino = np.linspace(0.5,6,300)
A_exata = 1/np.sqrt((k - m*w_fino**2)**2 + (c*w_fino)**2)
plt.plot(omegas,A_num,'bo',w_fino,A_exata,'r')
plt.axvline(np.sqrt(k/m - c**2/(2*m**2)),color='k',ls='--')
plt.xlabel(r"$\omega$"), plt.ylabel(r"Amplitude $A(\omega)$"), plt.grid(True)
plt.legend(["Numérica","Fórmula",r"$\omega_r$"])
plt.title("Curva de resposta em frequência")
plt.show()

### Oscilador de Van der Pol

A equação do [oscilador de Van der Pol](https://en.wikipedia.org/wiki/Van_der_Pol_oscillator) é dada por

$$
x'' - \mu(1 - x^2)x' + x = 0
$$

Escreva uma função chamada `van_der_pol` que receba os parâmetros `mu`, `u0` e `t` e retorne um array NumPy 2D de tamanho `(len(t),2)` com valores de $x$ na primeira coluna e valores de $x'$ na segunda coluna para a solução com valores iniciais `u0 = [x(0),x'(0)]`.

In [ ]:
def van_der_pol(mu,u0,t):

    def f(u,t):
        dudt = np.zeros(2)
        dudt[0] = u[1]
        dudt[1] = mu*(1 - u[0]**2)*u[1] - u[0]
        return dudt

    U = spi.odeint(f,u0,t)
    x = U[:,0]
    dxdt = U[:,1]
    return np.column_stack((x,dxdt))

Plote $x$ versus $x'$ para a solução com $\mu = 1$, $x(0) = 1$ e $x'(0)=4$.

In [ ]:
U = van_der_pol(1,[1,4],np.linspace(0,100,2500))
plt.plot(U[:,0],U[:,1]), plt.grid(True)
plt.xlabel("$x$"), plt.ylabel("$x'$")
plt.show()

Plote $x$ versus $x'$ para $x(0) = 1$ e $x'(0)=v_0$ para cada valor $v_0 = 1,2,3,4,5,6$ e $\mu=1$.

In [ ]:
for v0 in [1,2,3,4,5,6]:
    U = van_der_pol(1,[1,v0],np.linspace(0,10,500))
    plt.plot(U[:,0],U[:,1])
plt.title("Equação de Van der Pol")
plt.grid(True), plt.legend(['$v_0=1$','$v_0=2$','$v_0=3$','$v_0=4$','$v_0=5$','$v_0=6$'])
plt.show()

Plote $x$ versus $x'$ para $x(0) = 1$ e $x'(0)=4$ para cada valor $\mu = 0.2,0.5,1,2,3,4$.

In [ ]:
for mu in [0.2,0.5,1,2,3,4]:
    U = van_der_pol(mu,[1,4],np.linspace(0,10,500))
    plt.plot(U[:,0],U[:,1])
plt.title("Equação de Van der Pol")
plt.grid(True), plt.legend([r'$\mu=0.2$',r'$\mu=0.5$',r'$\mu=1$',r'$\mu=2$',r'$\mu=3$',r'$\mu=4$'])
plt.show()

Note que, para qualquer condição inicial (diferente da origem), as trajetórias se aproximam de uma mesma curva fechada: um **ciclo limite**. Esse comportamento é típico de sistemas não lineares e não ocorre em sistemas lineares.

### Modelo Predador-Presa de Lotka-Volterra

O modelo de [Lotka-Volterra](https://en.wikipedia.org/wiki/Lotka%E2%80%93Volterra_equations) descreve a interação entre uma população de presas $x(t)$ e de predadores $y(t)$:

\begin{align}
x' &= \alpha x - \beta xy \\\
y' &= \delta xy - \gamma y
\end{align}

* $\alpha$: taxa de crescimento das presas na ausência de predadores;
* $\beta$: taxa de predação;
* $\delta$: eficiência na conversão de presas em novos predadores;
* $\gamma$: taxa de mortalidade dos predadores na ausência de presas.

Igualando as derivadas a zero, obtemos os equilíbrios $(0,0)$ e $(\gamma/\delta,\ \alpha/\beta)$. Vamos usar $\alpha = 1$, $\beta = 0.1$, $\delta = 0.075$, $\gamma = 1.5$, para os quais o equilíbrio de coexistência é $(20, 10)$, e a condição inicial $x(0)=10$, $y(0)=5$.

In [ ]:
alpha = 1; beta = 0.1; delta = 0.075; gamma = 1.5

def lotka_volterra(u,t):
    x, y = u
    return [alpha*x - beta*x*y, delta*x*y - gamma*y]

t = np.linspace(0,30,2000)
U = spi.odeint(lotka_volterra,[10,5],t)
plt.plot(t,U[:,0],t,U[:,1]), plt.grid(True)
plt.xlabel("$t$"), plt.ylabel("População"), plt.legend(["Presas $x$","Predadores $y$"])
plt.title("Lotka-Volterra: séries temporais")
plt.show()

As populações oscilam periodicamente, com o pico de predadores ocorrendo logo depois do pico de presas. No plano de fase, isso aparece como órbitas fechadas em torno do equilíbrio $(20,10)$:

In [ ]:
X, Y = np.meshgrid(np.linspace(0.5,60,30),np.linspace(0.5,40,30))
plt.figure(figsize=(7,5))
plt.streamplot(X,Y,alpha*X - beta*X*Y,delta*X*Y - gamma*Y,color='lightgray',density=1.2)
for y0 in [3,5,8]:
    U = spi.odeint(lotka_volterra,[20,y0],t)
    plt.plot(U[:,0],U[:,1],lw=2,label=f"$x(0)=20$, $y(0)={y0}$")
plt.plot(gamma/delta,alpha/beta,'r*',markersize=12,label="Equilíbrio")
plt.xlabel("Presas $x$"), plt.ylabel("Predadores $y$"), plt.legend()
plt.title("Lotka-Volterra: retrato de fase")
plt.show()

Diferentemente do Van der Pol, aqui cada condição inicial gera uma órbita fechada **diferente**: não há ciclo limite. Isso acontece porque o sistema tem uma quantidade conservada,

$$
H(x,y) = \delta x - \gamma \ln x + \beta y - \alpha \ln y,
$$

constante ao longo de cada trajetória (verifique derivando $H(x(t),y(t))$ em relação a $t$!).

### Órbitas Planetárias

Dado um objeto com massa $M$, a magnitude da força da gravidade que ele exerce sobre outro objeto com massa $m$ é

$$
F = \frac{GMm}{d^2}
$$

onde $d$ é a distância entre os objetos. Vamos considerar a trajetória de um planeta de massa $m$ enquanto ele orbita uma estrela de massa $M$ (fixa no espaço). Vamos usar as seguintes unidades:

* massa solar (múltiplos da massa do Sol)
* anos
* [unidades astronômicas](https://en.wikipedia.org/wiki/Astronomical_unit) (aproximadamente a distância da Terra ao Sol)

Com essas unidades, a [constante gravitacional](https://en.wikipedia.org/wiki/Gravitational_constant) $G$ é $4 \pi^2$.

Seja $\mathbf{p} = (x,y)$ a posição do planeta em órbita com a estrela fixa na origem $(0,0)$, e seja $m_P$ a massa do planeta e $m_S$ a massa da estrela. Começando com a [Segunda Lei de Newton](https://en.wikipedia.org/wiki/Newton%27s_laws_of_motion#Newton.27s_second_law), temos

$$
m_P \frac{d^2 \mathbf{p}}{dt^2} = \mathbf{F}
$$

onde $\mathbf{F}$ é o vetor da força total atuando no planeta (que neste caso é apenas devido à gravidade). A [lei da gravitação universal de Newton](https://en.wikipedia.org/wiki/Newton%27s_law_of_universal_gravitation) afirma

$$
\mathbf{F} = - \frac{G m_S m_P}{ || \mathbf{p} ||^2 } \frac{\mathbf{p}}{ || \mathbf{p} || }
$$

onde $|| \mathbf{p} || = \sqrt{x^2 + y^2}$ é a distância entre a estrela e o planeta, e $-\mathbf{p} / || \mathbf{p} ||$ é o vetor unitário que aponta na direção do planeta para a estrela.

Juntando essas equações, obtemos um sistema bidimensional de equações diferenciais de segunda ordem

\begin{align}
\frac{d^2 x}{dt^2} &= - \frac{G m_S x }{ (x^2 + y^2)^{3/2} } \\\
\frac{d^2 y}{dt^2} &= - \frac{G m_S y }{ (x^2 + y^2)^{3/2} }
\end{align}

Escreva o sistema como um sistema de primeira ordem introduzindo novas variáveis: $u_0 = x$, $u_1 = x'$, $u_2 = y$ e $u_3 = y'$. O sistema se torna

\begin{align}
u_0' &= u_1 \\\
u_1' &= - \frac{G m_S u_0 }{ (u_0^2 + u_2^2)^{3/2} } \\\
u_2' &= u_3 \\\
u_3' &= - \frac{G m_S u_2 }{ (u_0^2 + u_2^2)^{3/2} }
\end{align}

Plote a órbita para $m_S = 1$ com as condições iniciais $x(0) = 1, x'(0) = 0, y(0) = 0, y'(0) = 7$.

In [ ]:
G = 4*np.pi**2; mS = 1;

def f(u,t):
    dudt = np.zeros(4)
    dudt[0] = u[1]
    dudt[1] = -G*mS*u[0]/(u[0]**2 + u[2]**2)**(3/2)
    dudt[2] = u[3]
    dudt[3] = -G*mS*u[2]/(u[0]**2 + u[2]**2)**(3/2)
    return dudt

u0 = [1,0,0,7]
t = np.linspace(0,4,2000)
U = spi.odeint(f,u0,t)
plt.plot(U[:,0],U[:,2],0,0,'r*')
plt.xlabel('$x$ (UA)'), plt.ylabel('$y$ (UA)')
plt.axis('equal'), plt.grid(True)
plt.show()

### Problema dos 3 Corpos de Euler

O [problema dos 3 corpos de Euler](https://en.wikipedia.org/wiki/Euler%27s_three-body_problem) é uma versão simplificada (e, admitidamente, fisicamente impossível) do [problema geral dos 3 corpos](https://en.wikipedia.org/wiki/Three-body_problem). O problema de Euler considera duas estrelas fixas no espaço e um planeta orbitando as estrelas em 2 dimensões. Vamos derivar as equações de movimento do planeta e, em seguida, plotar as trajetórias usando o solucionador de EDOs do SciPy, `odeint`.

Use as seguintes unidades:

* massa solar (múltiplos da massa do Sol)
* anos
* [unidades astronômicas (UA)](https://en.wikipedia.org/wiki/Astronomical_unit) (aproximadamente a distância da Terra ao Sol)

Com essas unidades, a constante gravitacional é $G = 4 \pi^2$. Introduza variáveis para o planeta e as estrelas:

* $m_{S_1}$ - massa da estrela 1
* $m_{S_2}$ - massa da estrela 2
* $m_P$ - massa do planeta
* $x_{S_1}$ - posição $x$ (fixa) da estrela 1
* $y_{S_1}$ - posição $y$ (fixa) da estrela 1
* $x_{S_2}$ - posição $x$ (fixa) da estrela 2
* $y_{S_2}$ - posição $y$ (fixa) da estrela 2
* $x_P$ - posição $x$ do planeta
* $y_P$ - posição $y$ do planeta
* $\mathbf{p} = (x_P,y_P)$ - vetor posição do planeta

Seja $\mathbf{F_1}$ a força da gravidade da estrela 1 atuando no planeta, e seja $\mathbf{F_2}$ a força da gravidade da estrela 2 atuando no planeta. A [Lei da Gravitação Universal de Newton](https://en.wikipedia.org/wiki/Newton%27s_law_of_universal_gravitation) afirma:

$$
\mathbf{F_1} = - \frac{ G m_P m_{S_1} }{ || \mathbf{d_1} ||^2} \frac{ \mathbf{d_1} }{ || \mathbf{d_1} || }
$$

$$
\mathbf{F_2} = - \frac{ G m_P m_{S_2} }{ || \mathbf{d_2} ||^2} \frac{ \mathbf{d_2} }{ || \mathbf{d_2} || }
$$

onde $\mathbf{d_1} = (x_P-x_{S_1},y_P-y_{S_1})$ é o vetor da estrela 1 para o planeta, e $\mathbf{d_2} = (x_P-x_{S_2},y_P-y_{S_2})$ é o vetor da estrela 2 para o planeta.

A [Segunda Lei do Movimento de Newton](https://en.wikipedia.org/wiki/Newton%27s_laws_of_motion) afirma:

$$
m_P \frac{ d^2 \mathbf{p} }{ dt^2 } = \mathbf{F}_1 + \mathbf{F}_2
$$

e, dividindo por $m_P$, obtemos ao sistema de EDOs de segunda ordem que governam o movimento do planeta:

$$
\frac{d^2 x_P}{dt^2} = - \frac{ G m_{S_1} (x_P - x_{S_1}) }{ || \mathbf{d_1} ||^3} - \frac{ G m_{S_2} (x_P - x_{S_2}) }{ || \mathbf{d_2} ||^3}
$$
$$
\frac{d^2 y_P}{dt^2} = - \frac{ G m_{S_1} (y_P - y_{S_1}) }{ || \mathbf{d_1} ||^3} - \frac{ G m_{S_2} (y_P - y_{S_2}) }{ || \mathbf{d_2} ||^3}
$$

Para plotar as trajetórias do planeta usando `odeint`, primeiro precisamos escrever o sistema como um sistema de primeira ordem. Introduza novas variáveis $u_0 = x_P$, $u_1 = x_P'$, $u_2 = y_P$ e $u_3 = y_P'$ e escreva

\begin{align}
u_0' &= u_1 \\\
u_1' &= - \frac{ G m_{S_1} (u_0 - x_{S_1}) }{ || \mathbf{d_1} ||^3} - \frac{ G m_{S_2} (u_0 - x_{S_2}) }{ || \mathbf{d_2} ||^3} \\\
u_2' &= u_3 \\\
u_3' &= - \frac{ G m_{S_1} (u_2 - y_{S_1}) }{ || \mathbf{d_1} ||^3} - \frac{ G m_{S_2} (u_2 - y_{S_2}) }{ || \mathbf{d_2} ||^3}
\end{align}

onde $\mathbf{d_1} = (u_0-x_{S_1},u_2-y_{S_1})$ e $\mathbf{d_2} = (u_0-x_{S_2},u_2-y_{S_2})$.

In [ ]:
G = 4*np.pi**2   # constante gravitacional (UA, anos, massas solares)
S1 = [-2,0]      # coordenadas da estrela 1
S2 = [2,0]       # coordenadas da estrela 2
M1 = 2           # massa da estrela 1 (em massas solares)
M2 = 1           # massa da estrela 2 (em massas solares)

def f(u,t):
    d1 = np.linalg.norm([u[0] - S1[0],u[2] - S1[1]])
    d2 = np.linalg.norm([u[0] - S2[0],u[2] - S2[1]])
    dudt = np.zeros(4)
    dudt[0] = u[1]
    dudt[1] = -G*M1*(u[0] - S1[0])/d1**3 - G*M2*(u[0] - S2[0])/d2**3
    dudt[2] = u[3]
    dudt[3] = -G*M1*(u[2] - S1[1])/d1**3 - G*M2*(u[2] - S2[1])/d2**3
    return dudt

u0 = [0,5,3,0]                # condições iniciais do planeta: [x, x', y, y']
t = np.linspace(0,30,2000)    # tempo (em anos)
u = spi.odeint(f,u0,t)        # u = [x, x', y, y']

plt.plot(u[:,0],u[:,2])                          # trajetória do planeta
plt.plot(S1[0],S1[1],'ro',markersize=5*M1)       # estrela 1 (círculo vermelho)
plt.plot(S2[0],S2[1],'ro',markersize=5*M2)       # estrela 2 (círculo vermelho)
plt.axis('equal')
plt.show()

Vamos colocar o código acima em uma função que podemos chamar com diferentes condições iniciais para ver que tipos de órbitas podemos criar!

In [ ]:
def euler3body(S1,S2,M1,M2,u0,tf,numpoints=1000):

    # lado direito do sistema de equações
    def f(u,t):
        d1 = np.linalg.norm([u[0]-S1[0],u[2]-S1[1]])
        d2 = np.linalg.norm([u[0]-S2[0],u[2]-S2[1]])
        dudt = np.zeros(4)
        dudt[0] = u[1]
        dudt[1] = -G*M1*(u[0]-S1[0])/d1**3 - G*M2*(u[0]-S2[0])/d2**3
        dudt[2] = u[3]
        dudt[3] = -G*M1*(u[2]-S1[1])/d1**3 - G*M2*(u[2]-S2[1])/d2**3
        return dudt

    t = np.linspace(0,tf,numpoints)   # tempo (em anos)
    u = spi.odeint(f,u0,t)            # u = [x, x', y, y']

    plt.plot(u[:,0],u[:,2])                          # trajetória do planeta
    plt.plot(S1[0],S1[1],'ro',markersize=5*M1)       # estrela 1
    plt.plot(S2[0],S2[1],'ro',markersize=5*M2)       # estrela 2
    plt.axis('equal')
    plt.show()

In [ ]:
euler3body([-1,0],[1,0],2,1,[0,5,3,0],30)

In [ ]:
euler3body([-2,0],[2,0],1.5,2.5,[0,4.8,3,0],20)

In [ ]:
euler3body([0,10],[0,-10],4.9,4.9,[0,np.pi,0,np.pi],100)